# Yellow Taxi Trip Records EDA

**Dataset:** NYC TLC Yellow Taxi Trip Records, January 2024  
**Source:** https://d37ci6vzurychx.cloudfront.net/trip-data  
**Data Dictionary:** https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

## Notebook structure

1. Introduction
2. Dataset overview
3. Initial profiling
4. Validation against the official TLC Data Dictionary
5. Exploratory Data Analysis
6. Final findings
7. Cleaning recommendations

## 1. Introduction

This notebook documents the exploratory analysis of the January 2024
NYC TLC Yellow Taxi Trip Records.

The purpose of the analysis is to understand the source data, compare
observed values with the official TLC documentation, test data-quality
hypotheses and produce evidence-based recommendations for the next
processing stage.

The raw source data is not overwritten in this notebook. Derived
analytical fields are created only to support the investigation.

## 2. Dataset overview

### 2.1 Setup and data loading

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

In [ ]:
FILE_PATH = Path("data/raw_download/yellow_tripdata_2024-01.parquet")

print(FILE_PATH)
print("File exists:", FILE_PATH.exists())

if not FILE_PATH.exists():
    raise FileNotFoundError(FILE_PATH)

file_size_mb = FILE_PATH.stat().st_size / 1024**2
print(f"Size: {file_size_mb:.2f} MB")

In [ ]:
df = pd.read_parquet(FILE_PATH)

### 2.2 Dataset structure

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(df.columns.tolist())
df.head()

In [ ]:
df.dtypes

### Initial structural conclusions

The dataset was loaded successfully and contains 2,964,624 rows and
19 columns. Timestamp fields were correctly parsed as datetimes.

Several identifier and categorical fields use numeric dtypes. The
`float64` dtype in some code columns is compatible with the presence
of missing values. `PULocationID` and `DOLocationID` are TLC Taxi Zone
identifiers, not geographic coordinates.

The documented value domains are validated before the main EDA.

## 3. Initial profiling

### 3.1 Missingness and basic profile

In [ ]:
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(2),
    "unique_including_na": df.nunique(dropna=False),
})

profile

In [ ]:
metric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
]

df[metric_columns].describe().T

The initial profile shows that missing values are concentrated in five
columns: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`,
`congestion_surcharge` and `Airport_fee`.

The numeric profile also shows several values requiring investigation,
including zero distances, negative monetary values and extreme maximum
values. These observations are investigated in the EDA sections below.

## 4. Validation against the official TLC Data Dictionary

The official TLC data dictionary is used as a reference for field
meanings, valid code values and payment semantics.

This validation does not clean or overwrite the source data. It only
compares the observed schema and values with the documented source
contract.

Important semantic points from the dictionary:

- `VendorID` identifies the TPEP provider.
- `RatecodeID = 99` means Null/unknown.
- `payment_type = 0` means Flex Fare.
- `tip_amount` contains automatically recorded credit-card tips;
  cash tips are not included.
- `total_amount` is the amount charged to the passenger and excludes
  cash tips.
- `PULocationID` and `DOLocationID` identify TLC Taxi Zones.

In [ ]:
PAYMENT_TYPES = {
    0: "Flex Fare",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

VENDOR_IDS = {
    1: "Creative Mobile Technologies, LLC",
    2: "Curb Mobility, LLC",
    6: "Myle Technologies Inc",
    7: "Helix",
}

RATE_CODES = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Null/unknown",
}

VALID_STORE_AND_FORWARD = {"Y", "N"}

def validate_numeric_domain(series, allowed_values):
    observed = set(series.dropna().astype(int))
    allowed = set(allowed_values)
    return {
        "observed": sorted(observed),
        "unknown_observed": sorted(observed - allowed),
        "all_observed_values_documented": observed <= allowed,
    }

domain_validation = pd.DataFrame({
    "VendorID": validate_numeric_domain(df["VendorID"], VENDOR_IDS),
    "RatecodeID": validate_numeric_domain(df["RatecodeID"], RATE_CODES),
    "payment_type": validate_numeric_domain(df["payment_type"], PAYMENT_TYPES),
}).T

domain_validation

In [ ]:
store_and_forward_validation = pd.Series({
    "non_null_values": sorted(df["store_and_fwd_flag"].dropna().unique()),
    "outside_Y_N": sorted(
        set(df["store_and_fwd_flag"].dropna().unique())
        - VALID_STORE_AND_FORWARD
    ),
    "missing_rows": int(df["store_and_fwd_flag"].isna().sum()),
})

store_and_forward_validation

### Taxi Zone and passenger-count checks

In [ ]:
UNKNOWN_ZONES = {264, 265}
MAX_ZONE_ID = 265

location_checks = pd.Series({
    "PULocationID missing": df["PULocationID"].isna().sum(),
    "DOLocationID missing": df["DOLocationID"].isna().sum(),
    "PULocationID <= 0": (df["PULocationID"] <= 0).sum(),
    "DOLocationID <= 0": (df["DOLocationID"] <= 0).sum(),
    "PULocationID > 265": (df["PULocationID"] > MAX_ZONE_ID).sum(),
    "DOLocationID > 265": (df["DOLocationID"] > MAX_ZONE_ID).sum(),
    "unique pickup locations": df["PULocationID"].nunique(dropna=True),
    "unique dropoff locations": df["DOLocationID"].nunique(dropna=True),
    "PU unknown-zone sentinels": df["PULocationID"].isin(UNKNOWN_ZONES).sum(),
    "DO unknown-zone sentinels": df["DOLocationID"].isin(UNKNOWN_ZONES).sum(),
})

location_checks

In [ ]:
passenger_checks = pd.Series({
    "missing passenger_count": df["passenger_count"].isna().sum(),
    "passenger_count = 0": (df["passenger_count"] == 0).sum(),
    "passenger_count between 1 and 6": df["passenger_count"].between(1, 6).sum(),
    "passenger_count > 6": (df["passenger_count"] > 6).sum(),
    "passenger_count < 0": (df["passenger_count"] < 0).sum(),
})

passenger_checks

### Data dictionary validation — conclusions

The observed values of `VendorID`, `RatecodeID` and `payment_type` are
documented by TLC. `VendorID = 6` is valid, `RatecodeID = 99` is a
documented Null/unknown sentinel and `payment_type = 0` is documented
as Flex Fare.

Non-null `store_and_fwd_flag` values belong to the documented `{Y, N}`
domain. Location IDs pass basic range checks; the meaning of sentinel
zones requires the separate TLC Taxi Zone reference.

## 5. Exploratory Data Analysis

### 5.1 Missing values and the structured missingness pattern

**Observation**

The same five columns contain 140,162 missing values, and
`payment_type = 0` occurs exactly 140,162 times.

**Hypothesis H1**

These conditions describe the same structured segment rather than
independent random missing values.

**Validation**

The following checks test whether all conditions identify exactly the
same rows and whether those rows form one contiguous segment.

In [ ]:
special_pattern_checks = pd.DataFrame({
    "passenger_count_missing": df["passenger_count"].isna(),
    "RatecodeID_missing": df["RatecodeID"].isna(),
    "store_and_fwd_flag_missing": df["store_and_fwd_flag"].isna(),
    "congestion_surcharge_missing": df["congestion_surcharge"].isna(),
    "Airport_fee_missing": df["Airport_fee"].isna(),
    "payment_type_zero": df["payment_type"].eq(0),
})

special_mask = special_pattern_checks.all(axis=1)
partial_special_mask = (
    special_pattern_checks.any(axis=1) & ~special_mask
)

print("Rows matching each condition:")
print(special_pattern_checks.sum())
print("\nRows matching all conditions:", special_mask.sum())
print("Rows matching only part of the pattern:", partial_special_mask.sum())

In [ ]:
special_positions = np.flatnonzero(special_mask.to_numpy())
is_contiguous = np.all(np.diff(special_positions) == 1)

pd.Series({
    "first special position": special_positions[0],
    "last special position": special_positions[-1],
    "special rows": len(special_positions),
    "contiguous block": is_contiguous,
})

In [ ]:
record_group = pd.Series(
    np.where(
        special_mask,
        "special_missing_group",
        "standard_group",
    ),
    index=df.index,
    name="record_group",
)

total_sign = pd.Series(
    np.select(
        [df["total_amount"] < 0, df["total_amount"].eq(0)],
        ["negative", "zero"],
        default="positive",
    ),
    index=df.index,
    name="total_sign",
)

pd.crosstab(record_group, total_sign, margins=True)

**Conclusion H1**

All six conditions identify the same 140,162 rows. The rows form one
contiguous block at the end of the file. This is strong evidence of a
distinct source segment or delivery format, but the trip records alone
cannot prove its provenance.

The rows are retained for analysis. Missing surcharge values are not
automatically interpreted as zero.

### 5.2 Monetary values

**Initial observation**

Negative values occur in multiple monetary columns. The number of
negative `total_amount` records is close to the number of records with
a negative `improvement_surcharge`.

**Hypothesis H2**

The main negative-fare/negative-total population is a coherent signed
transaction population rather than random corruption.

**Hypothesis H3**

Records with only a negative tip may represent corrections to recorded
card tips. This must be tested against `payment_type` rather than
assumed.

In [ ]:
anomaly_conditions = {
    "trip_distance = 0": df["trip_distance"].eq(0),
    "trip_distance > 100": df["trip_distance"].gt(100),
    "fare_amount < 0": df["fare_amount"].lt(0),
    "total_amount < 0": df["total_amount"].lt(0),
    "tip_amount < 0": df["tip_amount"].lt(0),
    "passenger_count = 0": df["passenger_count"].eq(0),
    "fare_amount = 0": df["fare_amount"].eq(0),
    "total_amount = 0": df["total_amount"].eq(0),
}

anomaly_summary = pd.DataFrame([
    {
        "condition": name,
        "count": int(condition.sum()),
        "percentage": round(condition.mean() * 100, 4),
    }
    for name, condition in anomaly_conditions.items()
]).sort_values("percentage", ascending=False)

anomaly_summary

#### Visual checks for selected distributions

In [ ]:
distance_plot_limit = df["trip_distance"].quantile(0.995)

df.loc[
    df["trip_distance"].between(0, distance_plot_limit),
    "trip_distance",
].hist(bins=100)

plt.title("Trip distance distribution — up to the 99.5th percentile")
plt.xlabel("Trip distance (miles)")
plt.ylabel("Number of trips")
plt.show()

In [ ]:
total_lower = df["total_amount"].quantile(0.005)
total_upper = df["total_amount"].quantile(0.995)

df.loc[
    df["total_amount"].between(total_lower, total_upper),
    "total_amount",
].hist(bins=100)

plt.title("Total amount distribution — central 99%")
plt.xlabel("Total amount")
plt.ylabel("Number of trips")
plt.show()

In [ ]:
negative_fare = df["fare_amount"] < 0
negative_total = df["total_amount"] < 0
negative_tip = df["tip_amount"] < 0

negative_populations = pd.Series({
    "fare < 0 and total < 0": (negative_fare & negative_total).sum(),
    "fare < 0 and total >= 0": (negative_fare & ~negative_total).sum(),
    "fare >= 0 and total < 0": (~negative_fare & negative_total).sum(),
    "tip < 0 only": (
        negative_tip & ~negative_fare & ~negative_total
    ).sum(),
})

payment_type_by_population = pd.DataFrame({
    "negative_fare_and_total": (
        df.loc[negative_fare & negative_total, "payment_type"]
        .map(PAYMENT_TYPES).value_counts()
    ),
    "negative_fare_or_total_mismatch": (
        df.loc[negative_fare ^ negative_total, "payment_type"]
        .map(PAYMENT_TYPES).value_counts()
    ),
    "negative_tip_only": (
        df.loc[
            negative_tip & ~negative_fare & ~negative_total,
            "payment_type",
        ].map(PAYMENT_TYPES).value_counts()
    ),
}).fillna(0).astype(int)

negative_populations

In [ ]:
payment_type_by_population

**Conclusion H2/H3**

Negative monetary values must be interpreted as signed transaction
data, not automatically converted to positive values or removed.
The payment-type distributions are retained as evidence for the
interpretation of the negative populations.

A negative tip alone is treated as a separate population. The notebook
does not call it a correction unless its payment-type distribution
supports that interpretation.

#### Monetary reconciliation — Hypothesis H4

The trip record dictionary defines `total_amount` as the amount
charged to the passenger. We test whether it reconciles with the
available monetary components within one cent.

This is an analytical consistency test, not a claim that the official
dictionary explicitly guarantees exact arithmetic for every row.

In [ ]:
amount_components = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
]

complete_amount_mask = df[amount_components].notna().all(axis=1)
complete_component_sum = df[amount_components].sum(
    axis=1,
    min_count=len(amount_components),
)
amount_residual = (
    df["total_amount"] - complete_component_sum
).round(2)

reconciliation = pd.DataFrame({
    "rows": [complete_amount_mask.sum()],
    "matching_within_1_cent": [
        amount_residual.loc[complete_amount_mask].abs().le(0.01).sum()
    ],
    "matching_percentage": [
        round(
            amount_residual.loc[complete_amount_mask]
            .abs().le(0.01).mean() * 100,
            4,
        )
    ],
    "maximum_absolute_difference": [
        amount_residual.loc[complete_amount_mask].abs().max()
    ],
})

reconciliation

In [ ]:
known_component_sum = df[amount_components].sum(axis=1, skipna=True)
special_residual = (
    df["total_amount"] - known_component_sum
).round(2)

special_residual_counts = (
    special_residual.loc[special_mask]
    .value_counts()
    .rename("count")
    .to_frame()
)
special_residual_counts["percentage"] = (
    special_residual_counts["count"]
    .div(special_mask.sum())
    .mul(100)
    .round(4)
)

special_residual_counts.head(15)

**Conclusion H4**

Reconciliation is evaluated separately for complete monetary records
and for the structured missing-data block. The residual of the special
block is diagnostic evidence that missing surcharge fields cannot be
safely replaced with zero.

In [ ]:
df["is_special_record"] = special_mask
df["is_negative_transaction"] = df["total_amount"] < 0
df["amount_residual"] = amount_residual
df["is_total_mismatch"] = (
    complete_amount_mask & amount_residual.abs().gt(0.01)
)

### 5.3 Dates, duration and speed

**Hypotheses**

- Negative or zero duration is temporally invalid.
- A concentration of durations close to 24 hours may indicate a
  timestamp or trip-closing problem.
- Extremely high speed is caused by a combination of implausible
  duration and/or distance values.

In [ ]:
df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

duration_quantiles = df["trip_duration_minutes"].quantile(
    [0, 0.001, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1]
)

duration_checks = pd.Series({
    "duration < 0": (df["trip_duration_minutes"] < 0).sum(),
    "duration = 0": df["trip_duration_minutes"].eq(0).sum(),
    "0 < duration < 1 minute": df["trip_duration_minutes"].between(
        0, 1, inclusive="neither"
    ).sum(),
    "duration > 6 hours": (df["trip_duration_minutes"] > 360).sum(),
    "duration > 24 hours": (df["trip_duration_minutes"] > 1440).sum(),
})

duration_quantiles

In [ ]:
duration_checks

In [ ]:
REPORTING_START = pd.Timestamp("2024-01-01")
REPORTING_END = pd.Timestamp("2024-02-01")
BOUNDARY_MARGIN = pd.Timedelta(days=7)
ANCIENT_CUTOFF = pd.Timestamp("2023-01-01")

pickup = df["tpep_pickup_datetime"]
outside_reporting_month = (
    (pickup < REPORTING_START) | (pickup >= REPORTING_END)
)
boundary_dates = outside_reporting_month & (
    (
        (pickup >= REPORTING_START - BOUNDARY_MARGIN)
        & (pickup < REPORTING_START)
    )
    | (
        (pickup >= REPORTING_END)
        & (pickup < REPORTING_END + BOUNDARY_MARGIN)
    )
)
ancient_or_untrustworthy_dates = (
    outside_reporting_month & (pickup < ANCIENT_CUTOFF)
)
unexplained_outside_dates = (
    outside_reporting_month
    & ~boundary_dates
    & ~ancient_or_untrustworthy_dates
)

date_checks = pd.Series({
    "earliest pickup": pickup.min(),
    "latest pickup": pickup.max(),
    "pickups outside January 2024": outside_reporting_month.sum(),
    "boundary-date candidates": boundary_dates.sum(),
    "ancient-date candidates": ancient_or_untrustworthy_dates.sum(),
    "unexplained outside dates": unexplained_outside_dates.sum(),
})

date_checks

In [ ]:
valid_duration_mask = df["trip_duration_minutes"] > 0
df["average_speed_mph"] = np.nan
df.loc[valid_duration_mask, "average_speed_mph"] = (
    df.loc[valid_duration_mask, "trip_distance"]
    / (df.loc[valid_duration_mask, "trip_duration_minutes"] / 60)
)

speed_quantiles = df["average_speed_mph"].quantile(
    [0, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1]
)

speed_checks = pd.Series({
    "speed = 0": df["average_speed_mph"].eq(0).sum(),
    "speed > 80 mph": (df["average_speed_mph"] > 80).sum(),
    "speed > 100 mph": (df["average_speed_mph"] > 100).sum(),
    "speed > 200 mph": (df["average_speed_mph"] > 200).sum(),
})

speed_quantiles.round(2)

In [ ]:
speed_checks

In [ ]:
high_speed_mask = df["average_speed_mph"] > 80

high_speed_diagnostics = pd.Series({
    "all high-speed records": high_speed_mask.sum(),
    "duration < 1 minute": (
        high_speed_mask & (df["trip_duration_minutes"] < 1)
    ).sum(),
    "trip_distance > 100 miles": (
        high_speed_mask & (df["trip_distance"] > 100)
    ).sum(),
    "trip_distance > 500 miles": (
        high_speed_mask & (df["trip_distance"] > 500)
    ).sum(),
    "special missing group": (
        high_speed_mask & special_mask
    ).sum(),
    "negative transaction": (
        high_speed_mask & df["is_negative_transaction"]
    ).sum(),
})

high_speed_diagnostics

In [ ]:
df.loc[
    df["average_speed_mph"].notna(),
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "average_speed_mph",
        "fare_amount",
        "total_amount",
        "is_special_record",
    ],
].sort_values("average_speed_mph", ascending=False).head(20)

In [ ]:
duration_ranges = pd.cut(
    df["trip_duration_minutes"],
    bins=[
        float("-inf"), 0, 1, 360, 1200, 1380, 1440, float("inf")
    ],
    labels=[
        "<= 0 min",
        "0-1 min",
        "1 min-6 h",
        "6-20 h",
        "20-23 h",
        "23-24 h",
        "> 24 h",
    ],
    right=True,
)

duration_range_counts = duration_ranges.value_counts().sort_index()
duration_range_counts

In [ ]:
near_24h_mask = df["trip_duration_minutes"].between(
    1380, 1440, inclusive="left"
)
hypothetical_duration = 1440 - df.loc[
    near_24h_mask, "trip_duration_minutes"
]
hypothetical_speed = df.loc[near_24h_mask, "trip_distance"] / (
    hypothetical_duration / 60
)

near_24h_diagnostic = pd.DataFrame({
    "original_duration_minutes": df.loc[
        near_24h_mask, "trip_duration_minutes"
    ],
    "hypothetical_duration_minutes": hypothetical_duration,
    "trip_distance": df.loc[near_24h_mask, "trip_distance"],
    "hypothetical_speed_mph": hypothetical_speed,
    "fare_amount": df.loc[near_24h_mask, "fare_amount"],
    "total_amount": df.loc[near_24h_mask, "total_amount"],
})

near_24h_diagnostic.describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
).T

In [ ]:
distance_by_group = pd.crosstab(
    record_group,
    pd.cut(
        df["trip_distance"],
        bins=[float("-inf"), 0, 50, 100, 500, float("inf")],
        labels=["<= 0", "0-50", "50-100", "100-500", "> 500"],
    ),
)

distance_by_group_percentages = (
    distance_by_group.div(distance_by_group.sum(axis=1), axis=0)
    .mul(100)
    .round(4)
)

distance_by_group_percentages

**Conclusions: dates, duration, distance and speed**

Most durations and speeds are plausible. The main anomalies are
concentrated in a small number of records with non-positive duration,
durations close to 24 hours, very short trips and extreme distances.

The near-24-hour diagnostic supports a timestamp or trip-closing
anomaly, but it is not sufficient evidence for automatically changing
timestamps. The special missing-data segment has substantially poorer
distance quality than the standard segment.

### 5.4 Duplicate checks

In [ ]:
exact_duplicate_rows = df.duplicated(keep=False).sum()
exact_duplicate_removable = df.duplicated(keep="first").sum()

pd.Series({
    "rows participating in exact duplicates": exact_duplicate_rows,
    "exact duplicate rows removable": exact_duplicate_removable,
})

In [ ]:
trip_identity_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount",
]

business_duplicate_rows = df.duplicated(
    subset=trip_identity_columns,
    keep=False,
).sum()
business_duplicate_removable = df.duplicated(
    subset=trip_identity_columns,
    keep="first",
).sum()

pd.Series({
    "rows participating in possible business duplicates": business_duplicate_rows,
    "possible business duplicate rows removable": business_duplicate_removable,
})

No exact duplicate rows or duplicates under the selected trip-identity
fields were found. Duplicate removal is therefore not an EDA finding
for this source file. A production ingestion pipeline may still keep a
duplicate safeguard because duplicates can be introduced during
ingestion or reruns.

## 6. Final findings

The EDA identified several distinct data-quality patterns:

1. A contiguous block of 140,162 records has the same missingness
   pattern across five fields and `payment_type = 0`.
2. The documented code domains are respected; `VendorID = 6`,
   `RatecodeID = 99` and `payment_type = 0` are valid documented values.
3. Negative monetary values form a signed transaction population that
   should not be silently converted or removed.
4. Missing surcharge values in the structured block must not be
   automatically replaced with zero.
5. A small group of records has invalid or suspicious temporal,
   distance or speed characteristics.
6. No exact or tested business duplicates were found.

The detailed observation–hypothesis–validation–conclusion table should
be saved as a separate Markdown cleaning contract in the next step.

## 7. Cleaning recommendations

The next project stage will implement the processing contract derived
from these findings.

The raw source values should remain preserved. The processing stage
should add explicit quality flags and create separate analytical
outputs where necessary. Pipeline implementation is intentionally kept
outside this EDA notebook.